# Core Intuition

1. **Model Initialization**
    
    Set baseline guess using the target mean:
    $$\hat{y}^{(0)} = \bar{y}$$
    
2. **Boosting Round $m$**
    
    **Step A: Gradients & Hessians**
    For MSE Loss $L = \frac{1}{2}(y_i - \hat{y}_i)^2$:
    - Gradient: $g_i = \hat{y}_i^{(m-1)} - y_i$
    - Hessian: $h_i = 1.0$
    
    **Step B: GOSS Sampling**
    1. Sort: Order samples by absolute gradient $\vert{}g_i\vert{}$ descending.
    2. Top Set $A$: Pick top $a\%$ samples with largest errors.
    3. Random Set $B$: Pick random $b\%$ from remaining samples.
    4. Rescale Factor: $W = \frac{1 - a}{b}$
    5. Adjust Gradients for Set $B$: $g_i \leftarrow W \cdot g_i$
    
    S**tep C: Split Finding & Leaf Weights**
    Using only GOSS samples ($A \cup B$):
    
    1. **Similarity Score:**
    $$\text{Sim} = \frac{\left(\sum g_i\right)^2}{\sum h_i + \lambda}$$
    
    2. **Gain**:
    $$\text{Gain} = \text{Sim}_L + \text{Sim}_R - \text{Sim}_{\text{Parent}}$$
    
    3. **Optimal Leaf Weight**:
    $$w = -\frac{\sum g_i}{\sum h_i + \lambda}$$
    
    **Step D: Update Prediction**
    For ALL dataset samples ($i = 1 \dots N$):
    $$\hat{y}_i^{(m)} = \hat{y}_i^{(m-1)} + \eta \cdot w$$
    
3. **Final Prediction**
$$\hat{y}_{\text{final}} = \bar{y} + \eta \cdot \sum_{m=1}^M w_m$$


# Mathematical Example

**Dataset**: 6 samples with feature $x$ and target $y$

- $x = [1, 2, 3, 4, 5, 6]$
- $y = [10, 14, 18, 42, 46, 50]$

**Hyperparameters:**

- Learning rate: $\eta = 0.5$
- Regularization: $\lambda = 1.0$
- GOSS parameters: $a = 0.33$ (Top 33% = 2 samples), $b = 0.50$ (Pick 50% from remaining 4 = 2 samples)

1. **Model Initialization**
    
    Calculate baseline target mean:
    $$\hat{y}^{(0)} = \bar{y} = \frac{10 + 14 + 18 + 42 + 46 + 50}{6} = \mathbf{30.0}$$
    
2. **Boosting Round $m = 1$**
    
    **Step A: Gradients & Hessians**
    
    Compute $g_i = \hat{y}_i^{(0)} - y_i$ and $h_i = 1.0$:
    
    |i|$x_i​$|Target $y_i$|​Guess $y_​i^{(0)}$|​Gradient $g_i$ |​Absolute Gradient $∣g_i​∣$| Hessian $h_i$|
    |---|---|---|---|---|---|---|
    |​1|1|10|30.0|$+20.0$|20.0|1.0|
    |2|2|14|30.0|$+16.0$|16.0|1.0|
    |3|3|18|30.0|$+12.0$|12.0|1.0|
    |4|4|42|30.0|$-12.0$|12.0|1.0|
    |5|5|46|30.0|$-16.0$|16.0|1.0|
    |6|6|50|30.0|$-20.0$|20.0|1.0|
    
    **Step B: GOSS Sampling**
    
    1. **Sort by $\vert{}g_i\vert{}$ descending:**
    Samples $\{1 (20.0), 6 (20.0), 2 (16.0), 5 (16.0), 3 (12.0), 4 (12.0)\}$.
     
    2. **Top Set $A$**($a = 33\% \implies 2$ samples): Pick top 2 largest error samples $\implies A = \{1, 6\}$.
    
    3. **Random Set $B$** ($b = 50\%$ from remaining 4 samples): Remaining pool is $\{2, 5, 3, 4\}$. Random selection picks $B = \{2, 5\}$. Samples $\{3, 4\}$ are dropped for this round.
    
    4. **Rescale Factor ($W$)**:
    $$W = \frac{1 - a}{b} = \frac{1 - 0.3333}{0.50} = \mathbf{1.3333} \quad \left(\text{or } \frac{4}{3}\right)$$
    
    5. **Adjust Gradients & Hessians for Set $B$**:

        - Sample 2 ($g_2 = +16.0$): $g_2^{\text{goss}} = 16.0 \times \frac{4}{3} = \mathbf{+21.333}$, $h_2^{\text{goss}} = 1.0 \times \frac{4}{3} = \mathbf{1.333}$
        - Sample 5 ($g_5 = -16.0$): $g_5^{\text{goss}} = -16.0 \times \frac{4}{3} = \mathbf{-21.333}$, $h_5^{\text{goss}} = 1.0 \times \frac{4}{3} = \mathbf{1.333}$
        
    **Step C: Split Finding & Leaf Weights**
    Evaluate candidate split $x \le 3.5$ using active GOSS samples $A \cup B = \{1, 2, 5, 6\}$:
    - **Left Leaf ($x \le 3.5$, Samples 1 & 2)**:
    $$\sum g_L = +20.0 + 21.333 = \mathbf{+41.333}, \quad \sum h_L = 1.0 + 1.333 = \mathbf{2.333}$$
    $$\text{Sim}_L = \frac{(+41.333)^2}{2.333 + 1.0} = \frac{1708.416}{3.333} = \mathbf{512.57}$$
    $$w_L = -\frac{+41.333}{2.333 + 1.0} = \mathbf{-12.40}$$
    
    - **Right Leaf ($x > 3.5$, Samples 5 & 6)**:
    $$\sum g_R = -21.333 - 20.0 = \mathbf{-41.333}, \quad \sum h_R = 1.333 + 1.0 = \mathbf{2.333}$$
    $$\text{Sim}_R = \frac{(-41.333)^2}{2.333 + 1.0} = \frac{1708.416}{3.333} = \mathbf{512.57}$$
    $$w_R = -\frac{-41.333}{2.333 + 1.0} = \mathbf{+12.40}$$
    
    **Step D: Update Predictions**
    Update predictions for ALL 6 dataset samples ($\eta = 0.5$):
    - For $x \le 3.5$ (Samples 1, 2, 3):
    $$\hat{y}^{(1)} = 30.0 + 0.5(-12.40) = \mathbf{23.80}$$
    - For $x > 3.5$ (Samples 4, 5, 6):
    $$\hat{y}^{(1)} = 30.0 + 0.5(+12.40) = \mathbf{36.20}$$
    
3. **Summary of Round 1 Results**

    |$x_i$|True Target $y_i$|Initial Guess $\hat y^{(0)}$|GOSS Role|Updated Prediction $\hat y^{(1)}$|Direction|
    |---|---|---|---|---|---|
    |1|10|30.0|Retained (Set $A$)|23.80|Shifted DOWN toward 10|
    |2|14|30.0|Sampled & Rescaled (Set $B$)|23.80|Shifted DOWN toward 14|
    |3|18|30.0|Dropped by GOSS|23.80|Updated via Left Leaf|
    |4|42|30.0|Dropped by GOSS|36.20|Updated via Right Leaf|
    |5|46|30.0|Sampled & Rescaled (Set $B$)|36.20|Shifted UP toward 46|
    |6|50|30.0|Retained (Set $A$)|36.20|Shifted UP toward 50|

    Now repeat the process 

# Python Implementation

In [3]:
import numpy as np

# -------------------------------------------------------------------
# LEAF-WISE TREE & NODE CLASSES
# -------------------------------------------------------------------


class LeafNode:

    def __init__(self, mask, g, h, depth=0, node_id=0):
        self.mask = mask
        self.g = g
        self.h = h
        self.depth = depth
        self.node_id = node_id
        self.is_leaf = True

        self.best_gain = -1.0
        self.best_feature = None
        self.best_threshold = None
        self.left_submask = None
        self.right_submask = None

        self.weight = 0.0
        self.left_child = None
        self.right_child = None


class LeafWiseTree:

    def __init__(
        self, max_leaves=4, reg_lambda=1.0, min_child_weight=0.1, gamma=0.0
    ):
        self.max_leaves = max_leaves
        self.reg_lambda = reg_lambda
        self.min_child_weight = min_child_weight
        self.gamma = gamma
        self.root = None
        self.next_node_id = 1

    def _calc_similarity(self, g, h):
        return (np.sum(g) ** 2) / (np.sum(h) + self.reg_lambda)

    def _calc_leaf_weight(self, g, h):
        return -np.sum(g) / (np.sum(h) + self.reg_lambda)

    def _find_best_split_for_leaf(self, node, X):
        X_leaf = X[node.mask]
        g_leaf, h_leaf = node.g, node.h
        n_samples, n_features = X_leaf.shape

        if n_samples <= 1 or np.sum(h_leaf) < self.min_child_weight:
            node.best_gain = -1.0
            return

        root_sim = self._calc_similarity(g_leaf, h_leaf)
        best_gain = -1.0
        best_split = None

        for f_idx in range(n_features):
            X_col = X_leaf[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))
            if len(sorted_unique) <= 1:
                continue

            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_m = X_col <= t
                right_m = ~left_m

                g_L, h_L = g_leaf[left_m], h_leaf[left_m]
                g_R, h_R = g_leaf[right_m], h_leaf[right_m]

                if (
                    np.sum(h_L) < self.min_child_weight
                    or np.sum(h_R) < self.min_child_weight
                ):
                    continue

                sim_L = self._calc_similarity(g_L, h_L)
                sim_R = self._calc_similarity(g_R, h_R)
                gain = sim_L + sim_R - root_sim - self.gamma

                if gain > best_gain:
                    best_gain = gain
                    best_split = (f_idx, t, left_m, right_m)

        if best_split is not None and best_gain > 0:
            node.best_gain = best_gain
            (
                node.best_feature,
                node.best_threshold,
                node.left_submask,
                node.right_submask,
            ) = best_split
        else:
            node.best_gain = -1.0

    def fit(self, X, g, h):
        root_mask = np.ones(X.shape[0], dtype=bool)
        self.root = LeafNode(
            root_mask, g, h, depth=0, node_id=self.next_node_id
        )
        self.root.weight = self._calc_leaf_weight(g, h)
        self.next_node_id += 1

        self._find_best_split_for_leaf(self.root, X)
        active_leaves = [self.root]

        current_leaves_count = 1

        print("=== STARTING LEAF-WISE TREE GROWTH ===")

        while current_leaves_count < self.max_leaves:
            # Display candidate pool of leaves and their potential gains
            print(
                f"\n--- Round {current_leaves_count}: Active Leaves Pool ---"
            )
            for leaf in active_leaves:
                print(
                    f"  Node ID {leaf.node_id} (Depth {leaf.depth}, Samples {np.sum(leaf.mask)}): "
                    f"Best Potential Gain = {leaf.best_gain:.4f}"
                )

            # SELECT THE SINGLE LEAF WITH MAXIMUM GAIN
            best_leaf_idx = -1
            max_gain = -1.0

            for idx, leaf in enumerate(active_leaves):
                if leaf.best_gain > max_gain:
                    max_gain = leaf.best_gain
                    best_leaf_idx = idx

            if best_leaf_idx == -1 or max_gain <= 0:
                print("\n[STOP] No leaf delivers Gain > 0. Stopping growth.")
                break

            target_leaf = active_leaves.pop(best_leaf_idx)
            target_leaf.is_leaf = False

            print(
                f"\n>>> SPLITTING WINNER: Node ID {target_leaf.node_id} "
                f"(Selected with Highest Gain = {target_leaf.best_gain:.4f}) on Feature {target_leaf.best_feature} <= {target_leaf.best_threshold:.2f}"
            )

            # Global boolean masks
            global_indices = np.where(target_leaf.mask)[0]
            left_global = global_indices[target_leaf.left_submask]
            right_global = global_indices[target_leaf.right_submask]

            left_mask = np.zeros(X.shape[0], dtype=bool)
            left_mask[left_global] = True

            right_mask = np.zeros(X.shape[0], dtype=bool)
            right_mask[right_global] = True

            left_child = LeafNode(
                left_mask,
                g[left_mask],
                h[left_mask],
                depth=target_leaf.depth + 1,
                node_id=self.next_node_id,
            )
            left_child.weight = self._calc_leaf_weight(
                g[left_mask], h[left_mask]
            )
            self.next_node_id += 1

            right_child = LeafNode(
                right_mask,
                g[right_mask],
                h[right_mask],
                depth=target_leaf.depth + 1,
                node_id=self.next_node_id,
            )
            right_child.weight = self._calc_leaf_weight(
                g[right_mask], h[right_mask]
            )
            self.next_node_id += 1

            target_leaf.left_child = left_child
            target_leaf.right_child = right_child

            self._find_best_split_for_leaf(left_child, X)
            self._find_best_split_for_leaf(right_child, X)

            active_leaves.append(left_child)
            active_leaves.append(right_child)

            current_leaves_count += 1

    def _predict_sample(self, x, node):
        if node.is_leaf:
            return node.weight
        if x[node.best_feature] <= node.best_threshold:
            return self._predict_sample(x, node.left_child)
        return self._predict_sample(x, node.right_child)

    def predict(self, X):
        return np.array([self._predict_sample(x, self.root) for x in X])


# -------------------------------------------------------------------
# TEST EXECUTION
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Multi-feature synthetic data with explicit non-linear targets
    np.random.seed(42)
    X_test = np.array(
        [
            [1.0, 10.0],
            [1.5, 12.0],
            [2.0, 50.0],
            [2.5, 55.0],
            [8.0, 100.0],
            [8.5, 105.0],
            [9.0, 200.0],
            [9.5, 205.0],
        ]
    )
    y_test = np.array([5.0, 7.0, 30.0, 32.0, 70.0, 72.0, 150.0, 152.0])

    # Initial mean baseline guess
    y_bar = np.mean(y_test)
    y_hat = np.full_like(y_test, y_bar)

    # MSE gradients & hessians
    g_init = y_hat - y_test
    h_init = np.ones_like(y_test)

    # Build tree with max 4 leaves
    tree = LeafWiseTree(max_leaves=4, reg_lambda=1.0)
    tree.fit(X_test, g_init, h_init)

    predictions = tree.predict(X_test)

    print("\n=== FINAL PREDICTIONS ===")
    for idx, (x_row, y_t, p_val) in enumerate(
        zip(X_test, y_test, predictions)
    ):
        print(f"Sample {idx+1} {x_row} | Target: {y_t:5.1f} | Leaf Weight: {p_val:7.2f}")

=== STARTING LEAF-WISE TREE GROWTH ===

--- Round 1: Active Leaves Pool ---
  Node ID 1 (Depth 0, Samples 8): Best Potential Gain = 14169.6429

>>> SPLITTING WINNER: Node ID 1 (Selected with Highest Gain = 14169.6429) on Feature 0 <= 8.75

--- Round 2: Active Leaves Pool ---
  Node ID 2 (Depth 1, Samples 6): Best Potential Gain = 2646.1905
  Node ID 3 (Depth 1, Samples 2): Best Potential Gain = -1.0000

>>> SPLITTING WINNER: Node ID 2 (Selected with Highest Gain = 2646.1905) on Feature 0 <= 5.25

--- Round 3: Active Leaves Pool ---
  Node ID 3 (Depth 1, Samples 2): Best Potential Gain = -1.0000
  Node ID 4 (Depth 2, Samples 4): Best Potential Gain = -1.0000
  Node ID 5 (Depth 2, Samples 2): Best Potential Gain = -1.0000

[STOP] No leaf delivers Gain > 0. Stopping growth.

=== FINAL PREDICTIONS ===
Sample 1 [ 1. 10.] | Target:   5.0 | Leaf Weight:  -37.00
Sample 2 [ 1.5 12. ] | Target:   7.0 | Leaf Weight:  -37.00
Sample 3 [ 2. 50.] | Target:  30.0 | Leaf Weight:  -37.00
Sample 4 [ 2.5 